# Ukrainian Folk Art Annotations: JSON to CSV

This notebook converts a Web Annotation JSON export into an analysis-ready CSV for the Ukrainian Folk Art dataset.

## Setup
Define a common `BASE_DIR` for file paths and, when running in Google Colab, optionally mount Google Drive.

In [1]:
from pathlib import Path



IN_COLAB = False
try:
  from google.colab import drive  # type: ignore
  IN_COLAB = True
except ImportError:
  pass

if IN_COLAB:
  # Optional in Colab if you need a newer package version:
  # !pip -q install pandas

  drive.mount('/content/drive')

USE_DRIVE = IN_COLAB  # set False to use local runtime files

if USE_DRIVE:
  BASE_DIR = Path('/content/drive/MyDrive/AISTER-Crowdsourcing-Pilot/step_3')
else:
  BASE_DIR = Path('../')

In [2]:
print(f'Base directory: {BASE_DIR.resolve()}')

Base directory: C:\Users\User\Desktop\Web2Learn\AISTER-Crowdsourcing-Pilot\step_3


## Configuration and input

Set input/output paths and load annotation JSON data.

In [3]:
import json

INPUT_JSON = BASE_DIR / 'data/5_ukrainian-folkart_annotations.json'
OUTPUT_CSV = BASE_DIR / 'outputs/5_ukrainian-folkart-annotations.csv'

with INPUT_JSON.open('r', encoding='utf-8') as f:
  data = json.load(f)

graph = data.get('@graph') or []
print(f'Loaded {len(graph)} annotations from {INPUT_JSON}')

Loaded 5946 annotations from ..\data\5_ukrainian-folkart_annotations.json


## Transform annotations

Build a normalized dataframe with safe defaults for nested fields.

In [4]:
import pandas as pd


def build_annotations_df(graph_items):
  rows = []

  for g in graph_items:
    source = (g.get('target') or {}).get('source') or ''

    rows.append({
      'created': g.get('created'),
      'value': (g.get('body') or {}).get('value'),
      'europeana_id': source.split('/')[-1],
      'upvotes': (g.get('review') or {}).get('upvotes') or 0,
      'downvotes': (g.get('review') or {}).get('downvotes') or 0,
      'recommendation': (g.get('review') or {}).get('recommendation') or 'unknown',
    })

  df_local = pd.DataFrame(rows)

  created_dt = pd.to_datetime(
    df_local['created'],
    format='%a %b %d %H:%M:%S UTC %Y',
    errors='coerce',
    utc=True,
  )

  df_local['created'] = created_dt.dt.strftime('%d.%m.%Y')
  df_local['upvotes'] = df_local['upvotes'].astype('int')
  df_local['downvotes'] = df_local['downvotes'].astype('int')

  df_local = df_local.sort_values(by='europeana_id').reset_index(drop=True)

  return df_local


df = build_annotations_df(graph)
display(df.head(30))

,created,value,europeana_id,upvotes,downvotes,recommendation
0,13.02.2026,cross,HMT1,7,3,accept
1,29.03.2026,Green background,HMT1,4,0,accept
2,13.02.2026,mary,HMT1,10,1,accept
3,13.02.2026,icon,HMT1,14,0,accept
4,27.03.2026,pair of icons,HMT1,9,0,accept
5,27.03.2026,Virgin Mary,HMT1,10,0,accept
6,27.03.2026,White lily,HMT1,9,0,accept
7,13.02.2026,religious icon,HMT1,13,0,accept
8,13.02.2026,wedding couple,HMT1,12,2,accept
9,29.03.2026,Wooden frame,HMT1,4,0,accept


Inspect core dataset stats before export.

In [5]:
print(f'Row count: {len(df)}')
print(df['recommendation'].value_counts(dropna=False))

Row count: 5946
recommendation
accept     5620
reject      259
unknown      67
Name: count, dtype: int64


## Export CSV

In [6]:
df.to_csv(OUTPUT_CSV, index=False)
print(f'Wrote: {OUTPUT_CSV}')

Wrote: ..\outputs\5_ukrainian-folkart-annotations.csv
